# YT Clipping: run from iPad

Turns a long YouTube video into vertical Shorts and sends them to Postiz.

**First time only:**
1. **Runtime → Change runtime type → T4 GPU** (makes transcription much faster).
2. Have your keys ready. Step 2 asks you to paste them (or save them once under the **🔑 Secrets** sidebar icon, if your screen shows it):
   - `POSTIZ_API_KEY`: from postiz.com → Settings → Public API
   - `HF_TOKEN` *(optional)*: Hugging Face token, after accepting the terms at huggingface.co/pyannote/speaker-diarization-3.0. Makes the crop follow the speaker.

3. *(Optional)* If downloads fail with "Sign in to confirm you're not a bot", export `cookies.txt` from a **spare** logged-in YouTube account and upload it in step 3.

Then fill in the form in step 4 and use **Runtime → Run all**.

## 1. Install (≈3–5 min)

In [ ]:
%cd /content
!rm -rf Yt-clipping- && git clone -q https://github.com/Gavin12715/Yt-clipping-.git
%cd /content/Yt-clipping-
!apt-get -qq install -y ffmpeg libmagic1 > /dev/null
!pip -q install -r requirements.txt
print("Installed")

## 2. Load secrets

In [ ]:
import os
from getpass import getpass

def secret(name):
    try:
        from google.colab import userdata
        return userdata.get(name)
    except Exception:
        return None

# Uses Colab Secrets (key icon) if set; otherwise asks you to paste it here.
os.environ["POSTIZ_API_KEY"] = secret("POSTIZ_API_KEY") or getpass("Paste your Postiz API key, then press Enter: ").strip()
os.environ["HF_TOKEN"] = secret("HF_TOKEN") or getpass("Hugging Face token (optional, press Enter to skip): ").strip()
os.environ["POSTIZ_URL"] = secret("POSTIZ_URL") or "https://api.postiz.com/public/v1"
print("Postiz key:", "set" if os.environ["POSTIZ_API_KEY"] else "MISSING")
print("HF token:", "set" if os.environ["HF_TOKEN"] else "not set (center crop)")

## 3. (Optional) Upload cookies.txt
Skip this cell unless downloads get blocked.

In [ ]:
from google.colab import files
up = files.upload()  # pick cookies.txt
if up:
    name = next(iter(up))
    os.rename(name, "cookies.txt")
    os.environ["YTDLP_COOKIES"] = os.path.abspath("cookies.txt")
    print("Cookies loaded")

## 4. Settings

In [ ]:
URL = "https://www.youtube.com/watch?v=VIDEO_ID"  #@param {type:"string"}
MAX_CLIPS = 5  #@param {type:"integer"}
MIN_SECONDS = 15  #@param {type:"integer"}
MAX_SECONDS = 60  #@param {type:"integer"}
WHISPER_MODEL = "small"  #@param ["tiny", "base", "small", "medium", "large-v2"]
ONLY_FROM = ""  #@param {type:"string"}
ONLY_TO = ""  #@param {type:"string"}
POST_MODE = "draft"  #@param ["none", "draft", "schedule", "now"]
PRIVACY = "public"  #@param ["public", "unlisted", "private"]
HOURS_BETWEEN_POSTS = 3  #@param {type:"number"}

## 5. Run

In [ ]:
import os
if os.path.exists("cookies.txt"):
    os.environ["YTDLP_COOKIES"] = os.path.abspath("cookies.txt")

from clipper.pipeline import run
results = run(
    URL, max_clips=MAX_CLIPS, min_dur=MIN_SECONDS, max_dur=MAX_SECONDS,
    whisper_model=WHISPER_MODEL,
    post=None if POST_MODE == "none" else POST_MODE,
    privacy=PRIVACY, every_minutes=int(HOURS_BETWEEN_POSTS * 60),
    start=ONLY_FROM or None, end=ONLY_TO or None,
)

## 6. Preview / download clips

In [ ]:
from IPython.display import Video, display
for r in results:
    print(f"{r['title']}  ({r['end'] - r['start']:.0f}s)")
    display(Video(r["file"], embed=True, width=270))

In [ ]:
# Download all clips as a zip to the iPad (Files app)
import shutil
from google.colab import files
shutil.make_archive("/content/clips", "zip", "clips")
files.download("/content/clips.zip")